# Dataset Distillation with Optimal Transport & PPDD on Kaggle
This notebook runs the complete distillation pipeline with zero download race conditions and automatic GPU configuration (supports 1x GPU or 2x T4 GPUs).

### Step 1: Environment Setup, Pretrained Models & Dataset Download

In [ ]:
import os, shutil, glob, torch
from torchvision import datasets

# 1. Clean workspace and clone repository
WORKDIR = '/kaggle/working/UROP'
if os.path.exists(WORKDIR):
    shutil.rmtree(WORKDIR)

!git clone https://github.com/AkmalMohammed-1/UROP.git {WORKDIR}
%cd {WORKDIR}

# 2. Install dependencies
!pip install -q efficientnet_pytorch PyYAML tqdm wandb

# 3. Pre-download CIFAR-10 in single-process mode (prevents DDP download race conditions)
print("Downloading / verifying CIFAR-10 dataset...")
os.makedirs(f"{WORKDIR}/dataset", exist_ok=True)
datasets.CIFAR10(f"{WORKDIR}/dataset", download=True, train=True)
datasets.CIFAR10(f"{WORKDIR}/dataset", download=True, train=False)
print(">>> CIFAR-10 ready! <<<")

# 4. Copy uploaded pretrained models from /kaggle/input if available
src_dirs = glob.glob('/kaggle/input/**/pretrained_models/cifar10', recursive=True)
if not src_dirs:
    src_dirs = glob.glob('/kaggle/input/**/cifar10', recursive=True)

dst_dir = f'{WORKDIR}/pretrained_models/cifar10'
os.makedirs(dst_dir, exist_ok=True)

if src_dirs:
    copied = 0
    for f in os.listdir(src_dirs[0]):
        if f.endswith('.pth.tar') or f.endswith('.pt'):
            shutil.copy2(os.path.join(src_dirs[0], f), os.path.join(dst_dir, f))
            copied += 1
    print(f">>> Loaded {copied} pretrained model checkpoints from Kaggle input! <<<")

# 5. Detect GPUs
num_gpus = torch.cuda.device_count()
gpu_ids = ','.join(str(i) for i in range(num_gpus))
print(f">>> Configured for {num_gpus} GPU(s): [{gpu_ids}] <<<")

### Step 2: Run Distillation with Optimal Transport (OT)

In [ ]:
%cd {WORKDIR}/condense

!torchrun --nproc_per_node={num_gpus} --nnodes=1 --master_port=29502 condense_script.py \
    --gpu={gpu_ids} \
    --ipc=10 \
    --config_path=../config/ipc10/cifar10.yaml

### Step 3: Run Evaluation

In [ ]:
import glob, os

# Locate generated checkpoint
checkpoints = glob.glob(f"{WORKDIR}/results/**/data_20000.pt", recursive=True)
if not checkpoints:
    checkpoints = glob.glob(f"{WORKDIR}/results/**/*.pt", recursive=True)

assert len(checkpoints) > 0, "Error: Distilled checkpoint not found! Make sure condensation finished."
latest_checkpoint = sorted(checkpoints, key=os.path.getmtime)[-1]
print(f"\n>>> Found Distilled Checkpoint: {latest_checkpoint} <<<")

%cd {WORKDIR}/evaluation

!torchrun --nproc_per_node=1 --nnodes=1 --master_port=29503 evaluation_script.py \
    --gpu=0 \
    --ipc=10 \
    --config_path=../config/ipc10/cifar10.yaml \
    --load_path="{latest_checkpoint}"